# GPG recovery for the $(3,3,1)$ BGM code

This notebook is now only a compact driver for the pulse-level recovery workflow.  The reusable implementation lives in `qer.gpgs`; noise channels come from `qer.noisemodel`; and Petz/Barnum--Knill recovery operators come from `qer.bk_recovery`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "python" / "codes").exists())
sys.path.insert(0, str(repo_root / "src"))

from qer import gpgs
from qer.bk_recovery import petz_recovery_kraus
from qer.codewords import bgmcode_kets_in_top_block
from qer.noisemodel import noisemodel


## Problem setup

We use the maximally mixed logical state
\[
\rho_C = \frac{1}{2}\left(|0_L\rangle\langle 0_L| + |1_L\rangle\langle 1_L|\right)
\]
as the input state.  The physical noise is the exact global symmetric amplitude-damping channel, while the four recovery Kraus operators are the Petz operators built from the second-order approximate global-AD Kraus map.


In [ ]:
b, g, m = 3, 3, 1
N = 2 * b * m + g
reference_weight = N
GPG_MODE = "noiseless"  # Change to "erroneous" to optimize with finite-cooperativity GPG errors.
GPG_COOPERATIVITY = np.inf  # Use a finite positive value when GPG_MODE == "erroneous".

ket0, ket1, _ = bgmcode_kets_in_top_block(b, g, m, return_qutip=True)
rho_c = (ket0 * ket0.dag() + ket1 * ket1.dag()) / 2


def exact_global_ad(p):
    return noisemodel(
        "global symmetric amplitude damping",
        N,
        float(p),
        1.0,
        return_rep="super",
        dynamics="exact",
    )


def approximate_petz_recovery(p):
    approx_kraus = noisemodel(
        "global symmetric amplitude damping",
        N,
        float(p),
        1.0,
        return_rep="kraus",
        dynamics="approx",
    )
    approx_kraus = gpgs.restrict_operators_to_dimension(approx_kraus, N + 1)
    return petz_recovery_kraus(approx_kraus, rho_c)

setup_summary = pd.DataFrame([
    {
        "code": "BGM (3,3,1)",
        "N": N,
        "system dimension": N + 1,
        "input state": "rho_C",
        "GPG reference": f"|D_{reference_weight}^{N}>",
        "GPG pulse mode": GPG_MODE,
        "recovery Kraus operators": len(approximate_petz_recovery(1e-3)),
    }
])
setup_summary


## Sweep

By default this cell loads the cached pulse synthesis results.  Set `RUN_OPTIMIZATION = True` to regenerate missing points; this can take a long time because each nontrivial rank-one phase target requires a GPG state-preparation optimization.  To optimize the pulses with the finite-cooperativity GPG error model, set `GPG_MODE = "erroneous"` and choose a finite `GPG_COOPERATIVITY`.


In [ ]:
P_SWEEP = np.logspace(-5, -2, 10)
RUN_OPTIMIZATION = False

if GPG_MODE == "noiseless":
    SWEEP_CACHE_PATH = repo_root / "datas" / "noiseless_gpgs_pulses" / "cache" / "gpg_exact_ad_sweep_cache.pkl"
else:
    cache_name = f"gpg_exact_ad_sweep_cache_{GPG_MODE}_C{GPG_COOPERATIVITY:.6g}.pkl".replace("+", "")
    SWEEP_CACHE_PATH = repo_root / "datas" / "noiseless_gpgs_pulses" / "cache" / cache_name
SWEEP_FIG_DIR = repo_root / "plots" / "AD"
SWEEP_FIG_BASENAME = "noiseless_gpg_implementation"

if RUN_OPTIMIZATION or not SWEEP_CACHE_PATH.exists():
    sweep = gpgs.run_gpg_recovery_sweep(
        rho_c,
        exact_global_ad,
        approximate_petz_recovery,
        P_SWEEP,
        logical_kets=(ket0, ket1),
        reference_weight=reference_weight,
        cache_path=SWEEP_CACHE_PATH,
        gpg_mode=GPG_MODE,
        cooperativity=GPG_COOPERATIVITY,
        log=print,
    )
    sweep_cache = sweep["cache"]
    sweep_results = sweep["results"]
else:
    sweep_cache = gpgs.load_gpg_sweep_cache(SWEEP_CACHE_PATH)
    sweep_results = gpgs.gpg_sweep_rows(sweep_cache).reset_index(drop=True)

sweep_results_display = sweep_results.copy()
for column in ["exact infidelity", "GPG infidelity", "GPG - exact infidelity penalty"]:
    sweep_results_display[f"{column} (sci)"] = sweep_results_display[column].map(lambda x: f"{x:.8e}")

sweep_results_display


## Figure and diagnostics


In [ ]:
fig_path = gpgs.redraw_gpg_recovery_sweep_figure(
    sweep_cache,
    SWEEP_FIG_DIR,
    basename=SWEEP_FIG_BASENAME,
)

fig, ax = plt.subplots(figsize=(3.45, 2.55), dpi=220)
ax.loglog(sweep_results["p"], sweep_results["exact infidelity"], "o-", lw=1.4, ms=4.2, label="exact recovery")
ax.loglog(sweep_results["p"], sweep_results["GPG infidelity"], "s--", lw=1.4, ms=4.0, label="GPG recovery")
ax.set_xlabel(r"$p = \gamma \Delta t$")
ax.set_ylabel(r"$1 - F$")
ax.grid(True, which="both", ls=":", lw=0.55, alpha=0.65)
ax.legend(frameon=True, fontsize=8)
fig.tight_layout()
print(f"Saved figure to {fig_path}")
fig


In [ ]:
sweep_synthesis_quality = pd.DataFrame([
    record
    for point in sweep_cache["points"].values()
    for record in point["synthesis_quality"]
])

sweep_optimization_summary = pd.DataFrame([
    {"p": point["metrics"]["p"], **record}
    for point in sweep_cache["points"].values()
    for record in point["optimization_records"]
])

sweep_pulse_table = pd.concat(
    [
        pd.DataFrame(X, columns=["alpha", "beta", "gamma", "kappa"]).assign(
            p=point["metrics"]["p"],
            factor=label,
            eig_index=eig_index,
            pulse=np.arange(1, len(X) + 1),
        )
        for point in sweep_cache["points"].values()
        for (label, eig_index), X in point["pulse_sequences"].items()
        if len(X) > 0
    ],
    ignore_index=True,
)[["p", "factor", "eig_index", "pulse", "alpha", "beta", "gamma", "kappa"]]

sweep_optimization_summary.sort_values("1 - F_state", ascending=False).head(20)


In [ ]:
sweep_synthesis_quality


In [ ]:
sweep_pulse_table